# 🧹 Olist E-commerce — Data Cleaning & Preparation
### Cleaning 9 raw relational tables into analysis-ready files with Python & Pandas
---
**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle)

**What this notebook does**
1. Loads the 9 raw CSV files **by name**
2. Audits each table (missing values, duplicates, data types, inconsistent text)
3. Standardises text and dates, and fixes order statuses that contradict their timestamps
4. Engineers delivery, cost and review features
5. Aggregates items, payments and reviews to **one row per order** and builds a master orders table
6. Adds coordinates to customers and sellers, **recording how each coordinate was matched** (`geo_match_level`)
7. Exports the cleaned files

> Scope: data cleaning and preparation only. No dashboards or modelling.

## 1. Import Libraries

In [1]:
import os
import glob
import unicodedata

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

#### Helper: Text Cleaning Function
Lowercases text, strips extra spaces and removes accents (e.g. `São Paulo` → `sao paulo`). It is applied to **all** city columns so city names match across tables.

In [2]:
def clean_text(text):
    if pd.isna(text):
        return text
    # Convert to string, lowercase, and strip spaces
    text = str(text).lower().strip()
    # Normalise and remove accents (e.g. são paulo -> sao paulo)
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    # Remove accidental double spaces
    return " ".join(text.split())

## 2. Load Datasets

In [3]:
# Folder that contains the raw Olist CSV files ("." = same folder as this notebook)
DATA_DIR = "."

# Load every CSV into a dictionary keyed by table name (e.g. "customers", "orders")
# "olist_orders_dataset.csv" -> "orders"; "olist_order_items_dataset.csv" -> "order_items"
tables = {}
for file in glob.glob(os.path.join(DATA_DIR, "*.csv")):
    name = os.path.basename(file).lower().replace(".csv", "")
    name = name.replace("olist_", "").replace("_dataset", "")
    tables[name] = pd.read_csv(file)

expected = ["customers", "geolocation", "orders", "order_items", "order_payments",
            "order_reviews", "products", "sellers", "product_category_name_translation"]
missing_tables = [t for t in expected if t not in tables]
assert not missing_tables, f"Missing files for: {missing_tables}. Found: {sorted(tables)}"

# Give each table its own clearly named variable
customers            = tables["customers"]
geo_raw              = tables["geolocation"]
orders               = tables["orders"]
order_items          = tables["order_items"]
order_payments       = tables["order_payments"]
order_reviews        = tables["order_reviews"]
products             = tables["products"]
sellers              = tables["sellers"]
category_translation = tables["product_category_name_translation"]

#### Check Dataset Shapes

In [4]:
# Check the shape of each table
for name in expected:
    print(f"{name:40s} shape = {tables[name].shape}")

customers                                shape = (99441, 5)
geolocation                              shape = (1000163, 5)
orders                                   shape = (99441, 8)
order_items                              shape = (112650, 7)
order_payments                           shape = (103886, 5)
order_reviews                            shape = (99224, 7)
products                                 shape = (32951, 9)
sellers                                  shape = (3095, 4)
product_category_name_translation        shape = (71, 2)


### Customer Table

#### Preview Data

In [5]:
customers.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


#### Inspect Data Types & Info

In [6]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


#### Check for Missing Values

In [7]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

#### Check for Duplicate Customer IDs

In [8]:
print("Number of Duplicated Rows in Customer Id are:",customers.duplicated(subset=['customer_id']).sum())

Number of Duplicated Rows in Customer Id are: 0


#### Explore Unique States & Cities

In [9]:
print("Unique states:")
print(customers['customer_state'].unique())

print("\nUnique cities (sample):")
print(customers['customer_city'].unique()[:50])

Unique states:
['SP' 'SC' 'MG' 'PR' 'RJ' 'RS' 'PA' 'GO' 'ES' 'BA' 'MA' 'MS' 'CE' 'DF'
 'RN' 'PE' 'MT' 'AM' 'AP' 'AL' 'RO' 'PB' 'TO' 'PI' 'AC' 'SE' 'RR']

Unique cities (sample):
['franca' 'sao bernardo do campo' 'sao paulo' 'mogi das cruzes' 'campinas'
 'jaragua do sul' 'timoteo' 'curitiba' 'belo horizonte' 'montes claros'
 'rio de janeiro' 'lencois paulista' 'caxias do sul' 'piracicaba'
 'guarulhos' 'pacaja' 'florianopolis' 'aparecida de goiania' 'santo andre'
 'goiania' 'cachoeiro de itapemirim' 'sao jose dos campos' 'sao roque'
 'camacari' 'resende' 'sumare' 'novo hamburgo' 'sao luis' 'sao jose'
 'santa barbara' 'ribeirao preto' 'ituiutaba' 'taquarituba'
 'sao jose dos pinhais' 'barrinha' 'parati' 'dourados' 'trindade'
 'cascavel' 'fortaleza' 'brasilia' 'pelotas' 'porto alegre' 'salto'
 'jundiai' 'cacapava' 'sao vicente' 'uberlandia' 'botelhos' 'sao goncalo']


#### Standardise City Names (Lowercase, Strip Whitespace & Remove Accents)

In [10]:
customers['customer_city'] = customers['customer_city'].apply(clean_text)

### Geolocation

#### Preview Data

In [11]:
geo_raw.head(3)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


#### Inspect Data Types & Info

In [12]:
geo_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


#### Check for Missing Values

In [13]:
geo_raw.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

#### Check for Duplicate Zip Code Prefixes

In [14]:
print("Number of Duplicated Rows in Geolocations are:",geo_raw.duplicated(subset=['geolocation_zip_code_prefix']).sum())

Number of Duplicated Rows in Geolocations are: 981148


#### Explore Unique States & Cities

In [15]:
print("Unique states:")
print(geo_raw['geolocation_state'].unique())

print("\nUnique cities (sample):")
print(geo_raw['geolocation_city'].unique()[:50])

Unique states:
['SP' 'RN' 'AC' 'RJ' 'ES' 'MG' 'BA' 'SE' 'PE' 'AL' 'PB' 'CE' 'PI' 'MA'
 'PA' 'AP' 'AM' 'RR' 'DF' 'GO' 'RO' 'TO' 'MT' 'MS' 'RS' 'PR' 'SC']

Unique cities (sample):
['sao paulo' 'são paulo' 'sao bernardo do campo' 'jundiaí'
 'taboão da serra' 'sãopaulo' 'sp' 'sa£o paulo' 'sao jose dos campos'
 'osasco' 'carapicuíba' 'carapicuiba' 'barueri' 'santana de parnaiba'
 'pirapora do bom jesus' 'santana de parnaíba' 'jandira' 'itapevi' 'cotia'
 'taboao da serra' 'vargem grande paulista' 'embu das artes'
 'itapecerica da serra' 'embu' 'são lourenço da serra'
 'sao lourenco da serra' 'embu-guacu' 'embu-guaçu' 'embu guaçu'
 'juquitiba' 'embu guacu' 'embuguacu' 'guarulhos' 'adamantina'
 'guarulhos-sp' 'aruja' 'arujá' 'santa isabel' 'mairipora' 'mairiporã'
 'cajamar' 'caieiras' 'jordanesia' 'polvilho' 'mauá' 'jordanésia'
 'franco da rocha' 'francisco morato' 'poa' 'itaquaquecetuba']


#### Clean City Names (Remove Accents & Normalise Text)

In [16]:
# Apply text cleaning to geolocation city names
geo_raw['geolocation_city'] = geo_raw['geolocation_city'].apply(clean_text)

#### Remove Coordinates Outside Brazil
The raw geolocation file contains points that fall outside Brazil. They would distort the averages, so they are dropped before flattening.

In [17]:
# Brazil's approximate bounding box
LAT_MIN, LAT_MAX = -33.75, 5.27
LNG_MIN, LNG_MAX = -73.99, -34.79

inside_brazil = (geo_raw['geolocation_lat'].between(LAT_MIN, LAT_MAX) &
                 geo_raw['geolocation_lng'].between(LNG_MIN, LNG_MAX))
print(f"Points outside Brazil removed: {(~inside_brazil).sum()} of {len(geo_raw)}")
geo_raw = geo_raw[inside_brazil].copy()

Points outside Brazil removed: 42 of 1000163


#### Deduplicate — Flatten to One Row per Zip Code Prefix

In [18]:
# Flatten the geolocation table to unique zip code prefixes
geolocation = geo_raw.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat', 'mean'),
    geolocation_lng=('geolocation_lng', 'mean'),
    geolocation_city=('geolocation_city', 'first'),
    geolocation_state=('geolocation_state', 'first')
).reset_index()

# Verify that there are now ZERO duplicate zip codes
print("Duplicate zip codes remaining:", geolocation['geolocation_zip_code_prefix'].duplicated().sum())

Duplicate zip codes remaining: 0


#### Verify & Preview Cleaned Geolocation Table

In [19]:
print(geolocation.shape)
geolocation.head(3)

(19010, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP


### Orders

#### Preview Data

In [20]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


#### Inspect Data Types & Info

In [21]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


#### Check for Missing Values

In [22]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

#### Check for Duplicate Order IDs

In [23]:
print("Number of Duplicated Rows in Order Id are:",orders.duplicated(subset=['order_id']).sum())

Number of Duplicated Rows in Order Id are: 0


#### Convert Timestamp Columns to Datetime

In [24]:
# List of timestamp columns
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Convert all of them to datetime objects
for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

#### Verify Data Types After Conversion

In [25]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

#### Explore Unique Order Statuses

In [26]:
orders.order_status.unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

#### Validate Missing Dates by Order Status

In [27]:
# 1. Check 'delivered' orders (Customer date should have 0 nulls)
delivered_orders = orders[orders['order_status'] == 'delivered']
print("--- Missing dates for DELIVERED orders ---")
print(delivered_orders[['order_delivered_customer_date', 'order_delivered_carrier_date']].isnull().sum())
print("\n")

# 2. Check 'shipped' orders (Carrier date should have 0 nulls)
shipped_orders = orders[orders['order_status'] == 'shipped']
print("--- Missing dates for SHIPPED orders ---")
print(shipped_orders[['order_delivered_carrier_date']].isnull().sum())
print("\n")

# 3. Check 'processing' orders (Should have approval date, but no carrier/customer dates)
processing_orders = orders[orders['order_status'] == 'processing']
print("--- Missing dates for PROCESSING orders ---")
print(processing_orders[['order_purchase_timestamp', 'order_approved_at']].isnull().sum())

--- Missing dates for DELIVERED orders ---
order_delivered_customer_date    8
order_delivered_carrier_date     2
dtype: int64


--- Missing dates for SHIPPED orders ---
order_delivered_carrier_date    0
dtype: int64


--- Missing dates for PROCESSING orders ---
order_purchase_timestamp    0
order_approved_at           0
dtype: int64


#### Isolate Anomalous Delivered Orders

In [28]:
# Isolate the 8 delivered orders that are missing a customer delivery date
delivered_anomalies = orders[
    (orders['order_status'] == 'delivered') & 
    ((orders['order_delivered_customer_date'].isnull()) | (orders['order_delivered_carrier_date'].isnull()))
]

# View the details of these specific orders
delivered_anomalies[['order_id', 'order_status', 'order_purchase_timestamp','order_approved_at', 'order_delivered_carrier_date','order_delivered_customer_date']]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT
73222,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT


#### Fix Incorrect Order Statuses Based on Date Availability

In [29]:
# Keep the original status so every correction is traceable
orders['order_status_original'] = orders['order_status']

# 1. status is delivered but customer date is missing AND carrier date exists -> change to 'shipped'
shipped_mask = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull()) & (orders['order_delivered_carrier_date'].notnull())
orders.loc[shipped_mask, 'order_status'] = 'shipped'

# 2. status is delivered but BOTH carrier and customer dates are missing -> change to 'approved'
approved_mask = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull()) & (orders['order_delivered_carrier_date'].isnull())
orders.loc[approved_mask, 'order_status'] = 'approved'

# Flag the corrected rows
orders['status_corrected'] = (orders['order_status'] != orders['order_status_original']).astype(int)
print("Orders with a corrected status:", orders['status_corrected'].sum())

Orders with a corrected status: 8


#### Verify Specific Anomaly Rows — Status Correction (looked up by order_id)

In [30]:
# Two orders that were 'delivered' without a customer date but had a carrier date -> now 'shipped'
orders[orders['order_id'].isin(['2d1e2d5bf4dc7227b3bfebb81328c15f', '0d3268bad9b086af767785e3f0fc0133'])][
    ['order_id', 'order_status_original', 'order_status', 'order_delivered_carrier_date', 'order_delivered_customer_date']]

,order_id,order_status_original,order_status,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,shipped,2017-11-30 18:12:23,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,shipped,2018-07-03 09:28:00,NaT


In [31]:
# An order that was 'delivered' with BOTH dates missing -> now 'approved'
orders[orders['order_id'] == '2d858f451373b04fb5c984a1cc2defaf'][
    ['order_id', 'order_status_original', 'order_status', 'order_delivered_carrier_date', 'order_delivered_customer_date']]

,order_id,order_status_original,order_status,order_delivered_carrier_date,order_delivered_customer_date
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,approved,NaT,NaT


#### Impute Missing Carrier Date Using Average Transit Time

In [32]:
# Avg of carrier shipping day
carrier_to_customer_days = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.days
avg_transit_days = round(carrier_to_customer_days.mean())

# specific anomaly row where carrier date is missing but customer date exists
impute_mask = (orders['order_status'] == 'delivered') & (orders['order_delivered_carrier_date'].isnull()) & (orders['order_delivered_customer_date'].notnull())

# Subtract the average transit days from the customer delivery date to fill the carrier date
orders.loc[impute_mask, 'order_delivered_carrier_date'] = orders.loc[impute_mask, 'order_delivered_customer_date'] - pd.Timedelta(days=avg_transit_days)

# Flag the imputed rows
orders['carrier_date_imputed'] = impute_mask.astype(int)
print("Carrier dates imputed:", orders['carrier_date_imputed'].sum())

Carrier dates imputed: 1


#### Confirm Imputation — No Remaining Nulls

In [33]:
orders[(orders['order_status'] == 'delivered') & 
           (orders['order_delivered_carrier_date'].isnull()) &
           (orders['order_delivered_customer_date'].notnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_status_original,status_corrected,carrier_date_imputed


#### Inspect Specific Anomaly Row After Fix

In [34]:
orders[orders['order_id'] == '2aa91108853cecb43c84a5dc5b277475'].T

,73222
order_id,2aa91108853cecb43c84a5dc5b277475
customer_id,afeb16c7f46396c0ed54acb45ccaaa40
order_status,delivered
order_purchase_timestamp,2017-09-29 08:52:58
order_approved_at,2017-09-29 09:07:16
order_delivered_carrier_date,2017-11-11 19:44:47
order_delivered_customer_date,2017-11-20 19:44:47
order_estimated_delivery_date,2017-11-14 00:00:00
order_status_original,delivered
status_corrected,0


#### Engineer Delivery Time Features

In [35]:
# Delivery Lead Time (Actual Delivery Time): How long the customer actually waited.
orders['delivery_lead_time_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

# Approval Delay: How long it took for the payment to be approved.
orders['approval_delay_hours'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600

# Seller Processing Time: How long the seller took to hand the item over to the logistics carrier.
orders['seller_processing_hours'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.total_seconds() /3600

# Carrier Shipping Time: How long the carrier took to deliver the item once picked up.
orders['carrier_shipping_days'] = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.days

# Estimated Delivery Accuracy (Delta): Did the package arrive early or late?
orders['delivery_delta_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

#### Preview Final Orders Table

In [36]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_status_original,status_corrected,carrier_date_imputed,delivery_lead_time_days,approval_delay_hours,seller_processing_hours,carrier_shipping_days,delivery_delta_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,delivered,0,0,8.0,0.178333,56.795833,6.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,delivered,0,0,13.0,30.713889,11.109167,12.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,delivered,0,0,9.0,0.276111,4.910278,9.0,-18.0


### Order Items

#### Preview Data

In [37]:
order_items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


#### Check for Duplicate Order Items

In [38]:
print("Number of Duplicated Rows in Order Item are:",order_items.duplicated(subset=['order_id','order_item_id']).sum())

Number of Duplicated Rows in Order Item are: 0


#### Check for Missing Values

In [39]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

#### Inspect Data Types & Info

In [40]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


#### Engineer Item-Level Cost & Freight Ratio Features

In [41]:
# Item-level total cost
order_items['item_total_cost'] = order_items['price'] + order_items['freight_value']

order_items['freight_ratio'] = order_items['freight_value'] / (order_items['price'] + order_items['freight_value'])

#### Convert Shipping Limit Date to Datetime

In [42]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

#### Preview Enriched Order Items Table

In [43]:
order_items.head(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total_cost,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,72.19,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,259.83,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,216.87,0.082400


#### Aggregate Order Items to Order Level

In [44]:
# Aggregate the order_items dataset to the order level
order_items_summary = order_items.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),                     # How many total items purchased
    unique_products=('product_id', 'nunique'),                  # How many distinct products
    total_sellers=('seller_id', 'nunique'),                    # How many different sellers involved
    order_price=('price', 'sum'),                               # Total item cost
    order_freight=('freight_value', 'sum'),                     # Total freight cost
    shipping_limit_deadline=('shipping_limit_date', 'max'),    # The final deadline for the order
).reset_index()


# add total order value and freight ratio columns
order_items_summary['total_order_value'] = (order_items_summary['order_price'] + order_items_summary['order_freight'])
order_items_summary['total_freight_ratio'] = (order_items_summary['order_freight'] / order_items_summary['total_order_value'])

# View the result
order_items_summary.head(3)

,order_id,total_items,unique_products,total_sellers,order_price,order_freight,shipping_limit_deadline,total_order_value,total_freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.9,13.29,2017-09-19 09:45:35,72.19,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.9,19.93,2017-05-03 11:05:13,259.83,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.0,17.87,2018-01-18 14:48:30,216.87,0.082400


### Payments

#### Preview Data

In [45]:
order_payments.head(3)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


#### Check for Duplicate Payment Records

In [46]:
print("Number of Duplicated Rows in Payments are:",order_payments.duplicated(subset=['order_id','payment_sequential']).sum())

Number of Duplicated Rows in Payments are: 0


#### Check for Missing Values

In [47]:
order_payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

#### Explore Unique Payment Types

In [48]:
order_payments.payment_type.unique()

array(['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'],
      dtype=object)

#### Aggregate Payments to Order Level

In [49]:
# Aggregate the payment dataset to the order level
order_payments_summary = order_payments.groupby('order_id').agg(
    total_payment_methods=('payment_sequential', 'count'),
    # This captures all unique payment types used in the order
    payment_method=('payment_type', lambda x: ', '.join(x.unique())),  
    max_installments=('payment_installments', 'max'),
    total_amount_paid=('payment_value', 'sum')
).reset_index()

# result
order_payments_summary.head(3)

,order_id,total_payment_methods,payment_method,max_installments,total_amount_paid
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,credit_card,3,259.83
2,000229ec398224ef6ca0657da4fc703e,1,credit_card,5,216.87


#### Verify Unique Payment Method Combinations

In [50]:
order_payments_summary.payment_method.unique()

array(['credit_card', 'boleto', 'voucher, credit_card', 'debit_card',
       'voucher', 'not_defined', 'credit_card, voucher',
       'credit_card, debit_card'], dtype=object)

### Reviews

#### Preview Data

In [51]:
order_reviews.head(3)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


#### Check for Duplicate Reviews

In [52]:
print("Number of Duplicated Rows in Reviews are:",order_reviews.duplicated(subset=['review_id','order_id']).sum())

Number of Duplicated Rows in Reviews are: 0


#### Check for Missing Values

In [53]:
order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

#### Handle Missing Comment Text — Fill Nulls & Create Flags

In [54]:
# 1. Fill actual NaN values with empty strings so text functions don't crash
order_reviews['review_comment_title'] = order_reviews['review_comment_title'].fillna('')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('')

# 2. Create flags: 1 if they wrote something, 0 if they left it blank
order_reviews['has_title'] = (order_reviews['review_comment_title'] != '').astype(int)
order_reviews['has_message'] = (order_reviews['review_comment_message'] != '').astype(int)

#### Flag Keyword Mentions — Delays, Quality Issues & Positive Praise

In [55]:
# Simple keyword flags (Portuguese). These are rough indicators, not sentiment analysis.
delay_pattern    = 'atraso|demorou|não entregue|nao entregue|não recebi|nao recebi'
quality_pattern  = 'defeito|estragado|quebrado|ruim'
positive_pattern = 'adorei|perfeito|ótimo|otimo|recomendo|rápido|rapido'

def mentions(pattern):
    """1 if the pattern appears in the review title OR message, else 0."""
    in_title   = order_reviews['review_comment_title'].str.contains(pattern, case=False)
    in_message = order_reviews['review_comment_message'].str.contains(pattern, case=False)
    return (in_title | in_message).astype(int)

order_reviews['mention_delay']    = mentions(delay_pattern)
order_reviews['mention_quality']  = mentions(quality_pattern)
order_reviews['mention_positive'] = mentions(positive_pattern)

#### Parse Review Dates & Calculate Response Time

In [56]:
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

order_reviews['review_response_days'] = (order_reviews['review_answer_timestamp'] - order_reviews['review_creation_date']).dt.days

#### Preview Enriched Reviews Table

In [57]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_title,has_message,mention_delay,mention_quality,mention_positive,review_response_days
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,,,2018-01-18,2018-01-18 21:46:59,0,0,0,0,0,0
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,,,2018-03-10,2018-03-11 03:05:13,0,0,0,0,0,1
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,,,2018-02-17,2018-02-18 14:36:24,0,0,0,0,0,1
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,0,1,0,0,0,0
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,0,1,0,0,1,1


#### Aggregate Reviews to Order Level

In [58]:
# order-level reviews summary
order_reviews_summary = order_reviews.groupby('order_id').agg(
    avg_review_score=('review_score', 'mean'),
    total_reviews_submitted=('review_id', 'count'),
    wrote_a_message=('has_message', 'max'),            # 1 if they wrote any text message
    mentions_delivery_delay=('mention_delay', 'max'),  # 1 if they complained about speed
    mentions_quality_issue=('mention_quality', 'max'),  # 1 if they complained about product quality
    mentions_positive_praise=('mention_positive', 'max') # 1 if they left positive keywords
).reset_index()

# View review summary
order_reviews_summary.head(3)

,order_id,avg_review_score,total_reviews_submitted,wrote_a_message,mentions_delivery_delay,mentions_quality_issue,mentions_positive_praise
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1,1,0,0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1,0,0,0,0
2,000229ec398224ef6ca0657da4fc703e,5.0,1,1,0,0,0


### Products

#### Preview Data

In [59]:
products.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


#### Check for Duplicate Products

In [60]:
print("Number of Duplicated Rows in Products are:",products.duplicated(subset=['product_id']).sum())

Number of Duplicated Rows in Products are: 0


#### Check for Missing Values

In [61]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

#### Fill Missing Category Names & Product Metadata

In [62]:
products_df = products.copy()

# 1. Fill missing category text with a placeholder string
products_df['product_category_name'] = products_df['product_category_name'].fillna('unknown')

# 2. Fill the missing lengths/photos with 0 (assuming no text/photos were provided)
products_df['product_name_lenght'] = products_df['product_name_lenght'].fillna(0)
products_df['product_description_lenght'] = products_df['product_description_lenght'].fillna(0)
products_df['product_photos_qty'] = products_df['product_photos_qty'].fillna(0)

#### Impute Missing Dimensions Using Category Median

In [63]:
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in dim_cols:
    # Median is less affected by very heavy / very large products than the mean
    products_df[col] = products_df.groupby('product_category_name')[col].transform(lambda x: x.fillna(x.median()))
    # Fallback: if a whole category has no value, use the overall median
    products_df[col] = products_df[col].fillna(products_df[col].median())

#### Engineer Product Volume Feature

In [64]:
products_df['product_volume_cm3'] = products_df['product_length_cm'] * products_df['product_height_cm'] * products_df['product_width_cm']

#### Confirm No Remaining Nulls

In [65]:
products_df.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
product_volume_cm3            0
dtype: int64

#### Load English Category Name Translation Table

In [66]:
df_english = category_translation
df_english.head(3)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


#### Merge English Category Names into Products Table

In [67]:
products_df = products_df.merge(df_english, on='product_category_name', how='left')

# Fill the English column for our 'unknown' placeholder
products_df['product_category_name_english'] = products_df['product_category_name_english'].fillna('unknown')

#### Confirm Nulls Resolved After Merge

In [68]:
products_df.isnull().sum()

product_id                       0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_volume_cm3               0
product_category_name_english    0
dtype: int64

### Sellers

#### Preview Data

In [69]:
sellers.head(3)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


#### Check for Duplicate Seller IDs

In [70]:
print("Number of Duplicated Rows in Sellers are:",sellers.duplicated(subset=['seller_id']).sum())

Number of Duplicated Rows in Sellers are: 0


#### Inspect Data Types & Info

In [71]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


#### Explore Unique States & Cities

In [72]:
print("Unique states:")
print(sellers['seller_state'].unique())

print("\nUnique cities (sample):")
print(sellers['seller_city'].unique()[:50])

Unique states:
['SP' 'RJ' 'PE' 'PR' 'GO' 'SC' 'BA' 'DF' 'RS' 'MG' 'RN' 'MT' 'CE' 'PB'
 'AC' 'ES' 'RO' 'PI' 'MS' 'SE' 'MA' 'AM' 'PA']

Unique cities (sample):
['campinas' 'mogi guacu' 'rio de janeiro' 'sao paulo' 'braganca paulista'
 'brejao' 'penapolis' 'curitiba' 'anapolis' 'itirapina' 'tubarao'
 'lauro de freitas' 'imbituba' 'brasilia' 'porto seguro' 'guaruja'
 'tabatinga' 'salto' 'tres de maio' 'belo horizonte' 'arapongas'
 'sao bernardo do campo' 'tatui' 'garopaba' 'camanducaia' 'tupa'
 'guarulhos' 'sao jose dos pinhais' 'sao ludgero' 'sao jose' 'piracicaba'
 'porto alegre' 'congonhal' 'santo andre' 'osasco' 'valinhos' 'joinville'
 'saquarema' 'barra velha' 'petropolis' 'santo antonio da patrulha'
 'ponta grossa' 'santana de parnaiba' 'sao carlos' 'ibitinga' 'barueri'
 'caxias do sul' 'araguari' 'contagem' 'jaragua do sul']


#### Standardise Seller City Names (Lowercase, Strip Whitespace & Remove Accents)

In [73]:
sellers['seller_city'] = sellers['seller_city'].apply(clean_text)

## 3. Build Master Orders Table
#### Merge Orders, Items, Payments & Reviews

In [74]:
# 1. Orders as the base
master_orders = orders.copy()

# 2. Sequential Left Joins to build the Master Table
master_orders = master_orders.merge(order_items_summary, on='order_id', how='left')
master_orders = master_orders.merge(order_payments_summary, on='order_id', how='left')
master_orders = master_orders.merge(order_reviews_summary, on='order_id', how='left')

#### Check for Missing Values in Master Table

In [75]:
master_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1782
order_delivered_customer_date    2965
order_estimated_delivery_date       0
order_status_original               0
status_corrected                    0
carrier_date_imputed                0
delivery_lead_time_days          2965
approval_delay_hours              160
seller_processing_hours          1796
carrier_shipping_days            2965
delivery_delta_days              2965
total_items                       775
unique_products                   775
total_sellers                     775
order_price                       775
order_freight                     775
shipping_limit_deadline           775
total_order_value                 775
total_freight_ratio               775
total_payment_methods               1
payment_method                      1
max_installm

#### Handle Missing Values — Zero Fill, Unknown & Null Flags

In [76]:
# Create a copy to prevent slicing warnings
master_orders_clean = master_orders.copy()

# 1. Financial, counts and keyword flags -> fill with 0
zero_fill_cols = [
    'total_items', 'unique_products', 'total_sellers', 
    'order_price', 'order_freight', 'total_order_value', 'total_freight_ratio',
    'total_payment_methods', 'max_installments', 'total_amount_paid',
    'total_reviews_submitted', 'wrote_a_message', 
    'mentions_delivery_delay', 'mentions_quality_issue', 'mentions_positive_praise'
]
master_orders_clean[zero_fill_cols] = master_orders_clean[zero_fill_cols].fillna(0)

# 2. Text/categorical columns -> fill with 'unknown'
master_orders_clean['payment_method'] = master_orders_clean['payment_method'].fillna('unknown')

# 3. avg_review_score is intentionally left as NaN for orders with no review
#    (a placeholder like 0 or -1 would distort averages)

# 4. Helper column to easily filter out non-delivered orders
master_orders_clean['is_delivered'] = master_orders_clean['order_delivered_customer_date'].notna().astype(int)

# Double check that the targeted nulls are gone
print(master_orders_clean[zero_fill_cols].isnull().sum())

total_items                 0
unique_products             0
total_sellers               0
order_price                 0
order_freight               0
total_order_value           0
total_freight_ratio         0
total_payment_methods       0
max_installments            0
total_amount_paid           0
total_reviews_submitted     0
wrote_a_message             0
mentions_delivery_delay     0
mentions_quality_issue      0
mentions_positive_praise    0
dtype: int64


## 4. Enrich Customer & Seller Tables with Geolocation
#### Primary Merge — Match by Zip Code Prefix
Every row records **how** its coordinates were found in `geo_match_level`: `zip` (most precise) → `city` (approximate) → `missing` (left empty on purpose).

In [77]:
geo_cols = ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']

# 1. Bring coordinates into the Customers table
customers_final = customers.merge(
    geolocation[geo_cols],
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
).drop(columns=['geolocation_zip_code_prefix'])
customers_final['geo_match_level'] = np.where(customers_final['geolocation_lat'].notna(), 'zip', 'missing')

# 2. Bring coordinates into the Sellers table
sellers_final = sellers.merge(
    geolocation[geo_cols],
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
).drop(columns=['geolocation_zip_code_prefix'])
sellers_final['geo_match_level'] = np.where(sellers_final['geolocation_lat'].notna(), 'zip', 'missing')

#### Check Missing Coordinates in Customers Table

In [78]:
customers_final.isnull().sum()

customer_id                   0
customer_unique_id            0
customer_zip_code_prefix      0
customer_city                 0
customer_state                0
geolocation_lat             279
geolocation_lng             279
geo_match_level               0
dtype: int64

#### Check Missing Coordinates in Sellers Table

In [79]:
sellers_final.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
geolocation_lat           7
geolocation_lng           7
geo_match_level           0
dtype: int64

#### Build City-Level Coordinate Backup Lookup
Average of all valid geolocation points in each city (cleaned names, Brazil-only points).

In [80]:
# Backup lookup: average coordinates of all valid points in each (city, state)
city_geo_backup = geo_raw.groupby(['geolocation_city', 'geolocation_state']).agg(
    backup_lat=('geolocation_lat', 'mean'),
    backup_lng=('geolocation_lng', 'mean')
).reset_index()

#### Secondary Patch — Fill Missing Coordinates Using City Average

In [81]:
# Patch Customers table
customers_final = customers_final.merge(
    city_geo_backup,
    left_on=['customer_city', 'customer_state'],
    right_on=['geolocation_city', 'geolocation_state'],
    how='left'
)

# Fill only rows that have no coordinates yet and a city match exists
city_mask = customers_final['geolocation_lat'].isna() & customers_final['backup_lat'].notna()
customers_final.loc[city_mask, 'geolocation_lat'] = customers_final.loc[city_mask, 'backup_lat']
customers_final.loc[city_mask, 'geolocation_lng'] = customers_final.loc[city_mask, 'backup_lng']
customers_final.loc[city_mask, 'geo_match_level'] = 'city'

# Drop the backup columns
customers_final.drop(columns=['geolocation_city', 'geolocation_state', 'backup_lat', 'backup_lng'], errors='ignore', inplace=True)
customers_final.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_lat,geolocation_lng,geo_match_level
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.498489,-47.396929,zip
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.727992,-46.542848,zip
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531642,-46.656289,zip


#### Secondary Patch — Fill Seller Missing Coordinates Using City Average

In [82]:
# Patch Sellers table
sellers_final = sellers_final.merge(
    city_geo_backup,
    left_on=['seller_city', 'seller_state'],
    right_on=['geolocation_city', 'geolocation_state'],
    how='left'
)

city_mask = sellers_final['geolocation_lat'].isna() & sellers_final['backup_lat'].notna()
sellers_final.loc[city_mask, 'geolocation_lat'] = sellers_final.loc[city_mask, 'backup_lat']
sellers_final.loc[city_mask, 'geolocation_lng'] = sellers_final.loc[city_mask, 'backup_lng']
sellers_final.loc[city_mask, 'geo_match_level'] = 'city'

sellers_final.drop(columns=['geolocation_city', 'geolocation_state', 'backup_lat', 'backup_lng'], errors='ignore', inplace=True)
sellers_final.head(3)

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_lat,geolocation_lng,geo_match_level
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893848,-47.061337,zip
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383437,-46.947927,zip
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909572,-43.177703,zip


#### Confirm Remaining Nulls After City-Level Patch

In [83]:
print('Null Values in Customers:')
print(customers_final.isnull().sum())
print('\nNull Values in Sellers:')
print(sellers_final.isnull().sum() )

Null Values in Customers:
customer_id                  0
customer_unique_id           0
customer_zip_code_prefix     0
customer_city                0
customer_state               0
geolocation_lat             58
geolocation_lng             58
geo_match_level              0
dtype: int64

Null Values in Sellers:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
geolocation_lat           0
geolocation_lng           0
geo_match_level           0
dtype: int64


#### Remaining Missing Coordinates — Intentionally Left Empty
The customers (and sellers) that still have no coordinates are **not** filled with a state average. A state average can be hundreds of km from the real location and would distort any distance or delivery-time analysis. They stay `NaN` and are labelled `geo_match_level = 'missing'`, so they are easy to filter.

In [84]:
print("Customers by geo_match_level:")
print(customers_final['geo_match_level'].value_counts())
print("\nSellers by geo_match_level:")
print(sellers_final['geo_match_level'].value_counts())

# Use only exact (zip) matches when precise distances are needed:
# customers_final[customers_final['geo_match_level'] == 'zip']

Customers by geo_match_level:
geo_match_level
zip        99162
city         221
missing       58
Name: count, dtype: int64

Sellers by geo_match_level:
geo_match_level
zip     3088
city       7
Name: count, dtype: int64


## Assumptions & Limitations
These choices change the data, so they are listed here for transparency:

- Orders marked `delivered` with **no customer delivery date** were re-labelled `shipped` (carrier date present) or `approved` (no carrier date). The original value is kept in `order_status_original`; corrected rows have `status_corrected = 1`.
- A missing carrier date on a delivered order was **estimated** as *customer delivery date − average carrier transit days*. Imputed rows have `carrier_date_imputed = 1`.
- Missing product dimensions were filled with the **category median** (overall median as a fallback); missing text/photo counts were filled with 0.
- Orders with no items, payments or reviews get 0 (or `unknown` for text) in the master table; `avg_review_score` stays empty when there is no review.
- Coordinates outside Brazil's bounding box were removed before averaging. Customer/seller coordinates are exact only when `geo_match_level = 'zip'`; `city` is an approximation and `missing` has no coordinates.
- Review keyword flags use a short Portuguese keyword list and are rough indicators only.

## 5. Export Cleaned Files
#### Export Master Orders Table

In [85]:
import os

# 1. Create the folder if it doesn't exist yet
os.makedirs('Clean Files', exist_ok=True)

# Export clean version to CSV
master_orders_clean.to_csv('Clean Files/Master_Order_Details.csv', index=False)

#### Export Supporting Clean Tables

In [86]:
# Export Other Tables

# 1. Order_items
order_items.to_csv('Clean Files/Order_Items_Granular.csv', index=False)

# 2. Products 
products_df.to_csv('Clean Files/Products_Cleaned.csv', index=False)

# 3. Customers
customers_final.to_csv('Clean Files/Customers_Cleaned.csv', index=False)

# 4. Sellers
sellers_final.to_csv('Clean Files/Sellers_Cleaned.csv', index=False)